Ok guys, this notebook is to get the functional data of the 3 coregistered sessions.

You get for each session 3 .csv: 
df_dff_...: cointating the actual dff's for every roi (and raw traces in case you want to do your own analysis).
df_epochs_...: a table that contains the epochs data (stimuli). Its basically the timestasmps of when each stimuli started.
df_roi_indexing_...: the table you need to match each ROI to the EM data.

the name of the files contain the column (_col#_) and the volume (_vol#_). This can be used to identify each session, as a specific column_volum imagining combination it's done in only one session.

Sorry if it's not the best code and there are a copule of hardcoded variables you need to pay attention to.




In [2]:
import sys

from os.path import join as pjoin
import platform

# Add the directory for the data and utilities
mat_version = 1196

platstring = platform.platform()
system = platform.system()
if system == "Darwin":
    # macOS
    data_root = "/Volumes/Brain2026/"
elif system == "Windows":
    # Windows (replace with the drive letter of USB drive)
    data_root = "E:/"
elif "amzn" in platstring:
    # then on CodeOcean
    data_root = "/data/"
else:
    # then your own linux platform
    # EDIT location where you mounted hard drive
    data_root = "/media/$USERNAME/Brain2026/"

# Set the directory to load prepared data and utility code
data_dir = pjoin(data_root, f"v1dd_{mat_version}")
functional_dir = pjoin(data_root, f"v1dd_{mat_version}_coreg_functional_correlation")
utils_dir = pjoin("..", "utils")

# Add utilities to path
sys.path.append(utils_dir)

In [3]:
import numpy as np
import pandas as pd
from pathlib import Path
import hdmf
import pynwb
from hdmf_zarr import NWBZarrIO

import matplotlib.pyplot as plt
%matplotlib inline 

## Session discovery and selection

Every session is uniquely indexed by the combination of its `(column, volume)`. 
For example:

> **`(column 1, volume 3)`, `(column 1, volume 5)`**

However, that is not the session ID name or load path. The `(column, volume)` information is save _within_ every NWB file. But to map between sessions, use the aggregated session index:



In [4]:
session_index = pd.read_csv(pjoin(functional_dir, "session_index.csv"))

# We can filter to only those coregistered sessions:
coreg_session_index = session_index.query('coregistered')
coreg_session_index

,path,name,format,column,volume,n_planes,session_id,coregistered
14,/data/409828_V1DD_Filtered/409828_2018-12-07_1...,409828_2018-12-07_15-05-38_filtered_2026-08-18...,zarr,2,5,6,791865124,True
22,/data/409828_V1DD_Filtered/409828_2018-12-13_1...,409828_2018-12-13_15-10-05_filtered_2026-04-09...,zarr,1,3,6,794964451,True
24,/data/409828_V1DD_Filtered/409828_2018-12-14_1...,409828_2018-12-14_14-47-35_filtered_2026-04-09...,zarr,1,5,6,795771997,True


In [5]:
# let's pull the first coregistered_session
coreg_example = 0 #HERE IS WHERE YOU SELECT WHICH COREGISTERD SESSION YOU WANT TO LOOK AT

session_name = coreg_session_index.iloc[coreg_example]['name']
session_nwb_path = coreg_session_index.iloc[coreg_example]['path']

print(session_name, ' | ',
      f"column={coreg_session_index.iloc[coreg_example].column}, volume={coreg_session_index.iloc[coreg_example].volume}", 
      f" ({coreg_session_index.iloc[coreg_example].session_id}) ",
     )

409828_2018-12-07_15-05-38_filtered_2026-08-18_23-10-02  |  column=2, volume=5  (791865124) 


In [ ]:
#read the session file
io = NWBZarrIO(session_nwb_path, mode = 'r') 
nwbfile_zarr = io.read()

In [ ]:
# this produces the dataframe of all ROIs in all Planes of the specific session (col_vol combination)
# look out for the hardcoded lines where you state the col/vol. 

plane_list = [0, 1, 2, 3, 4, 5]
data_list = []

for plane in plane_list:

    processing = nwbfile_zarr.processing[f'plane-{plane}']

    # Each array is frames × ROIs
    dff = processing['dff'].data[:]
    demixed = processing['demixed'].data[:]
    raw = processing['raw'].data[:]
    neuropil_corrected = processing['neuropil_corrected'].data[:]
    events = processing['events'].data[:]

    # Check that all arrays have the same dimensions
    assert dff.shape == demixed.shape == raw.shape == neuropil_corrected.shape == events.shape, \
        f"Shape mismatch in plane {plane}"

    # Create one row per ROI
    df = pd.DataFrame({
        'dff': list(dff.T),
        'demixed': list(demixed.T),
        'raw': list(raw.T),
        'neuropil_corrected': list(neuropil_corrected.T),
        'events': list(events.T),
        'plane': plane,
        'col': 1,       # HARDOCDED FOR NOW, MAYBE FIX IT OR MAKE IT BETTER
        'volume': 5,    # HARDOCDED FOR NOW, MAYBE FIX IT OR MAKE IT BETTER
    })

    # ROI ID within the plane
    df['roi'] = range(len(df))

    data_list.append(df)


# Combine all planes
dff_df = pd.concat(data_list, ignore_index=True)

print(dff_df.shape)
print(dff_df.head())

(906, 9)
                                                 dff  \
0  [1.1326565, 0.12610407, 1.4266396, 1.8803906, ...   
1  [0.54362833, 0.22379696, 0.42190823, 0.2495097...   
2  [0.871868, 0.7344089, 1.4147545, 0.8169179, 0....   
3  [1.5027747, 1.0496231, 1.1743734, 1.8210032, 1...   
4  [1.5082428, 0.7196614, 1.4080576, 1.8478926, 2...   

                                             demixed  \
0  [461.79367, 354.94885, 556.5546, 635.114, 671....   
1  [468.953, 447.00427, 503.99146, 495.62393, 529...   
2  [561.67224, 532.79596, 686.56525, 601.4114, 61...   
3  [548.0968, 494.6165, 567.4194, 679.73474, 654....   
4  [539.5662, 405.0525, 572.12805, 653.39197, 716...   

                                                 raw  \
0  [513.29816, 394.12424, 600.42236, 687.0745, 73...   
1  [468.953, 447.00427, 503.99146, 495.62393, 529...   
2  [561.67224, 532.79596, 686.56525, 601.4114, 61...   
3  [548.0968, 494.6165, 567.4194, 679.73474, 654....   
4  [622.1409, 502.05453, 653.5273, 75

In [ ]:
#to save the df, remmeber to change the name with correct col/vol number
#dff_df.to_csv('df_dff_col1_vol5.csv', index=False)

In [ ]:
# Get the roi indexing information for the example plane. Tip: this is critical for coregistering to EM structural data
roi_plane_info = nwbfile_zarr.processing[f'plane-{plane}']['dff'].rois.to_dataframe()
roi_plane_info

In [ ]:
#now this gets you the roi indexing info to match ROIs with EM
# remember to watch out with the hardcoded vol/col number.

plane_list = [0, 1, 2, 3, 4, 5]
roi_info_list = []

for plane in plane_list:

    # Get ROI information for this plane
    roi_plane_info = (
        nwbfile_zarr.processing[f'plane-{plane}']['dff']
        .rois
        .to_dataframe()
    )

    # Add plane information
    roi_plane_info['plane'] = plane

    roi_info_list.append(roi_plane_info)


# Combine all planes
roi_info_df = pd.concat(
    roi_info_list,
    ignore_index=True
)

print(roi_info_df.shape)
roi_info_df.head()

In [ ]:
#save the df 
#roi_info_df.to_csv('df_roi_indexing_col1_vol5.csv', index=False)
#to match ROIs to EM

In [9]:
#epoch table with the information of when the different stimuli regime starts
epoch_table = nwbfile_zarr.intervals['epochs'].to_dataframe()
epoch_table

,stim_name,start_time,stop_time,duration
id,,,,
0,drifting_gratings_full,51.881851,339.104187,287.222351
1,drifting_gratings_windowed,342.123352,629.345825,287.222473
2,locally_sparse_noise,632.364990,872.581665,240.216675
3,spontaneous,873.565796,1173.799072,300.233276
4,natural_images_12,1176.818237,1328.928223,152.109985
5,natural_movie,1340.954834,1791.313110,450.358276
6,locally_sparse_noise,1795.366455,2095.482910,300.116455
7,natural_images,2097.584717,2396.750488,299.165771
8,drifting_gratings_windowed,2411.846191,2699.068604,287.222412


In [ ]:
#save the epoch table
#epoch_table.to_csv('df_epochs_col1_vol5.csv', index=False)